# 🚀 PostgreSQL (pgvector) 기반 학술 논문 통합 임베딩 파이프라인 (심화 가이드)

본 노트북은 **BIST Mini-2 프로젝트**의 학술 RAG(Retrieval-Augmented Generation) 시스템에 대량의 논문 데이터를 안정적이고 비용 효율적으로 적재하기 위해 구축된 **심화 파이프라인**을 설명합니다.

실제 운영 환경의 DB 설정을 건드리지 않고 흐름을 검증할 수 있는 **드라이런(Dry-Run) 시뮬레이션 모드**를 지원하며, 주석으로 **실제 백엔드 서버에 반영되는 프로덕션 코드**가 완벽히 구성되어 있어 실무 응용에 최적화되어 있습니다.

---

## 📌 심화 파이프라인의 4가지 핵심 최적화 기법
1. **중복 적재 제거 (Deduplication)** : 이미 데이터베이스에 인서트되어 있는 `arxiv_id` 목록을 캐싱하여, 중복된 논문은 처리 대상에서 제외해 리소스를 절약합니다.
2. **아카이브 스냅샷 일자 수집** : arXiv 스냅샷 파일(`arxiv-metadata-oai-snapshot.json`)로부터 최종 문헌 업데이트 일자(`update_date`) 정보를 파싱하여 4-키 표준 메타데이터를 정교하게 완성합니다.
3. **임베딩 캐시 활용 어댑터 (OrderedPrecomputedEmbeddings)** : 기계산 임베딩 캐시 파일(`local_embeddings_no_chunk_output.jsonl`)이 존재하는 경우, 실제 OpenAI API 요금을 지불하지 않고 어댑터 클래스를 통해 데이터를 무비용으로 주입합니다.
4. **비동기 벌크 배치 처리 (Bulk Async)** : 대량 문헌을 `BATCH_SIZE` 단위로 비동기 루프로 전송하여 트랜잭션 락을 방지하고 적재 속도를 최대화합니다.

## [준비] 백엔드 환경 설정 및 모듈 로드

노트북 실행 위치를 파악하고 백엔드 패키지 경로를 주입한 뒤, 필수 환경변수(데이터베이스 URL 및 OpenAI API 키)를 로드합니다.

In [6]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 현재 notebooks/embedding_ingestion/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

# 백엔드 모듈 임포트를 위해 sys.path에 추가
if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# backend/.env 파일 자동 감지 및 환경 변수 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정(.env)을 성공적으로 로드했습니다. (경로: {env_path})")
else:
    print("ℹ️ 백엔드 .env 파일을 찾지 못해 시스템 환경 변수를 대신 사용합니다.")

# 환경 변수 로드
database_url = os.getenv("DATABASE_URL", "")
openai_key = os.getenv("OPENAI_API_KEY", "")

print(f"🔑 OpenAI API Key 설정 여부: {'설정됨(Yes)' if openai_key else '설정안됨(No)'}")
print(f"🗄️ Database URL: {database_url[:35]}..." if database_url else "🗄️ Database URL: 설정안됨")

✅ 백엔드 환경 설정(.env)을 성공적으로 로드했습니다. (경로: /Users/pileuszu/Repos/bist-mini-2/backend/.env)
🔑 OpenAI API Key 설정 여부: 설정됨(Yes)
🗄️ Database URL: postgresql+asyncpg://postgres:postg...


## 1단계. 원본 데이터 포맷 및 DB 테이블 구조 이해

임베딩 적재를 시작하기 전에, 우리가 다루는 입력 파일의 형태와 최종적으로 적재되는 데이터베이스의 컬럼 구조를 명확히 이해해야 합니다.

### 1.1 입력 데이터 포맷의 종류
1. **일반 논문 메타데이터 파일 (`bio_gn_embeddings.jsonl` 등)**
   * 한 줄에 하나의 JSON 객체가 기록된 줄바꿈 분리형 JSONL 파일로, 제목, 초록 등의 기본 정보를 포함합니다.
2. **기계산 임베딩 캐시 파일 (`local_embeddings_no_chunk_output.jsonl` 등)**
   * 이미 OpenAI API를 거쳐 `3072차원` 임베딩 벡터가 계산되어 있는 파일입니다. 이 파일이 있다면 API 비용을 지불하지 않습니다.
3. **arXiv 전체 메타데이터 스냅샷 파일 (`arxiv-metadata-oai-snapshot.json`)**
   * 전체 arXiv 논문의 메타데이터 아카이브로, 개별 데이터에 없는 최종 업데이트 일자(`update_date`) 정보를 조회하는 매핑 맵 구축용으로 사용합니다.

---

### 1.2 PostgreSQL pgvector 테이블 저장 구조
LangChain의 `PGVector` 라이브러리를 연동하면 내부적으로 다음의 2개 테이블이 사용됩니다.

#### ① `langchain_pg_collection` 테이블 (컬렉션 공간 관리)
* `name`: 컬렉션 이름 (예: cs_embeddings, bio_embeddings 등 학술 도메인 단위)
* `uuid`: 컬렉션 고유 ID (부모 테이블 키)

#### ② `langchain_pg_embedding` 테이블 (실제 데이터 및 임베딩)
* `collection_id`: 컬렉션 UUID (어떤 학술 도메인 공간에 속하는가)
* `embedding`: **vector(3072)** 타입. 3072차원 임베딩 벡터 실수값
* `document`: **TEXT** 타입. RAG 검색에 사용되는 본문 (`Title: 제목\n\nAbstract: 초록`)
* `cmetadata`: **JSONB** 타입. 필터링 검색에 활용되는 **BIST Mini-2 표준 4-키** (`title`, `arxiv_id`, `categories`, `update_date`)

## 2단계. DB 적재 정보 및 아카이브 스냅샷 일자 수집

파이프라인의 시작은 외부 정보 수집입니다.
1. 중복 저장을 방지하기 위해 **기존 DB에 적재된 arxiv_id 목록**을 수집합니다.
2. 최종 업데이트 정보가 유실되어 있을 때를 대비해 **메타데이터 스냅샷 맵**을 사전 구축합니다.

In [7]:
import json
from typing import Set, Dict, List
from pathlib import Path

# 2.1 기존 DB에 이미 들어있는 논문 ID 목록 조회
def get_existing_ids(collection_name: str) -> Set[str]:
    """실제 데이터베이스에 연결하여 이미 적재된 arxiv_id 목록을 반환합니다."""
    print(f"[STEP 1] DB에서 컬렉션 '{collection_name}'에 들어있는 cmetadata->>'arxiv_id' 값들을 조회합니다.")
    import psycopg
    existing_ids = set()
    sync_conn_str = database_url.replace("postgresql+psycopg_async://", "postgresql://").replace("postgresql+asyncpg://", "postgresql://")
    with psycopg.connect(sync_conn_str) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT uuid FROM langchain_pg_collection WHERE name = %s", (collection_name,))
            row = cur.fetchone()
            if row:
                collection_uuid = row[0]
                cur.execute("SELECT cmetadata->>'arxiv_id' FROM langchain_pg_embedding WHERE collection_id = %s", (collection_uuid,))
                for r in cur.fetchall():
                    if r[0]:
                        existing_ids.add(r[0])
    print(f"        -> [DB 조회 완료] 현재 기적재된 arxiv_id 목록: {existing_ids}")
    return existing_ids

# 2.2 메타데이터 스냅샷에서 업데이트 일자 매핑 정보 생성
def load_update_date_map() -> Dict[str, str]:
    """arxiv-metadata-oai-snapshot.json 아카이브 파일을 스캔하여 ID별 update_date 매핑 테이블을 생성합니다."""
    print(f"[STEP 2] 스냅샷 아카이브 파일을 스캔하여 update_date 매핑 정보를 빌드합니다...")
    snapshot_path = Path("../../data/raw/archive/arxiv-metadata-oai-snapshot.json")
    mapping = {}
    if not snapshot_path.exists():
        print(f"경고: {snapshot_path} 파일이 없어 update_date를 공란으로 채웁니다.")
        return mapping
    with open(snapshot_path, "r", encoding="utf-8") as f:
        for line in f:
            if "cs.NE" not in line:
                continue
            try:
                data = json.loads(line)
                categories = data.get("categories", "")
                if "cs.NE" in categories.split():
                    mapping[data["id"]] = data.get("update_date", "")
            except (json.JSONDecodeError, KeyError):
                continue
    print(f"        -> [스냅샷 빌드 완료] 총 {len(mapping)}개의 날짜 정보가 준비되었습니다.")
    return mapping

existing_ids = get_existing_ids("cs_embeddings")
update_date_map = load_update_date_map()

[STEP 1] DB에서 컬렉션 'cs_embeddings'에 들어있는 cmetadata->>'arxiv_id' 값들을 조회합니다.
        -> [DB 조회 완료] 현재 기적재된 arxiv_id 목록: {'2404.07143', '2104.04121', '1811.10636', '2010.07078', 'cs/0505069', '2209.15007', '2211.08522', '2201.11729', '2306.04525', '2606.05408', '1910.01763', '2511.14775', '1504.02366', '2109.04459', '2401.02429', '2305.19468', '2212.10078', '2308.12864', '2501.04832', '2303.11860', '2208.04832', '1704.02681', '1309.3214', '2112.11018', '2108.08691', '2401.07387', '2009.01527', '2210.09171', '2011.13585', '2510.13757', '1709.04057', '2602.03967', '2004.12750', '2107.07382', '1207.4448', '2510.04595', '2212.02233', '2409.12846', '1108.4199', '1808.03920', '1412.1602', '2211.03481', '1911.10113', '2002.00027', '2007.03547', '1911.05371', '2307.13007', '2511.18429', '2412.20090', '2404.07724', '2110.10969', '2309.15090', '2010.09531', '2411.00803', '2108.04707', '2108.12684', '2506.08749', '2202.04890', '1707.06132', 'cs/0502007', '2003.00063', '1809.08860', '2504.17801', '260

## 3단계. 신규 문서 선별 및 4-키 표준 가공

준비된 중복 방지 세트(`existing_ids`)와 날짜 매핑 맵(`update_date_map`)을 바탕으로 입력 데이터를 정제합니다.
1. **중복 검사**: 기적재 ID에 존재하면 **스킵(SKIP)**하여 데이터 중복을 방지합니다.
2. **본문 결합**: RAG 검색에서 제목과 초록이 함께 벡터화될 수 있도록 `Title: 제목\n\nAbstract: 요약` 형식의 단일 본문 텍스트를 만듭니다.
3. **표준 메타데이터 생성**: 4가지 필수 필드(`title`, `arxiv_id`, `categories`, `update_date`)를 메타데이터 딕셔너리로 조립합니다.

In [8]:
from langchain_core.documents import Document

def process_and_standardize_documents(raw_data: List[dict], existing_ids: Set[str], date_map: Dict[str, str]):
    """입력 데이터를 받아 중복을 거르고 표준 도큐먼트로 가공하는 파이프라인의 중심 필터부입니다."""
    print(f"[STEP 3] 신규 논문 필터링 및 4-키 표준 메타데이터 적용 시작...")
    
    standardized_docs = []
    precomputed_embeddings = []
    
    for i, paper in enumerate(raw_data):
        doc_id = paper.get("id") or paper.get("arxiv_id") or ""
        print(f"\n    * 처리 대상 문서 #{i+1} (ID: {doc_id})")
        
        # 1) 중복 제거 필터링
        if doc_id in existing_ids:
            print(f"      [SKIP] -> 이미 기존 DB에 등록된 arxiv_id이므로 이번 적재 대상에서 스킵합니다.")
            continue
            
        # 2) RAG 검색용 본문 결합
        title = (paper.get("title") or "").replace("\n", " ").strip()
        abstract = (paper.get("abstract") or "").replace("\n", " ").strip()
        content = f"Title: {title}\n\nAbstract: {abstract}"
        
        # 3) 4-키 표준 메타데이터 포맷 구성
        metadata = {
            "title": title,
            "arxiv_id": doc_id,
            "categories": paper.get("categories", ""),
            "update_date": date_map.get(doc_id, "") # 동적 매핑 병합
        }
        
        # LangChain 표준 Document 객체 생성
        doc = Document(page_content=content, metadata=metadata)
        standardized_docs.append(doc)
        
        # precomputed 캐시 임베딩 보관
        precomputed_embeddings.append(paper.get("cached_embedding"))
        print(f"      [ADD]  -> 중복 없음 확인! RAG 본문 및 4-키 메타데이터 구성 완료.")
        
    return standardized_docs, precomputed_embeddings


# 시뮬레이션용 논문 원본 리스트 (ID: 2401.00001은 중복 데이터, 2401.00002는 캐시가 있는 데이터, 2401.00003은 실시간 API 대상)
mock_raw_papers = [
    {
        "id": "2401.00001",
        "title": "Deep Learning in Biology",
        "abstract": "This paper presents a new model for genetic analysis...",
        "categories": "q-bio.GN",
        "cached_embedding": [0.01] * 3072
    },
    {
        "id": "2401.00002",
        "title": "Sequence Alignment Algorithms",
        "abstract": "Fast sequence alignment methods are critical for genomic study...",
        "categories": "cs.NE",
        "cached_embedding": [-0.02] * 3072
    },
    {
        "id": "2401.00003",
        "title": "Quantum Astro-Computing",
        "abstract": "Quantum computational techniques applied to astrophysical datasets...",
        "categories": "astro-ph.EP",
        "cached_embedding": None
    }
]

documents, cached_embeddings = process_and_standardize_documents(mock_raw_papers, existing_ids, update_date_map)

[STEP 3] 신규 논문 필터링 및 4-키 표준 메타데이터 적용 시작...

    * 처리 대상 문서 #1 (ID: 2401.00001)
      [ADD]  -> 중복 없음 확인! RAG 본문 및 4-키 메타데이터 구성 완료.

    * 처리 대상 문서 #2 (ID: 2401.00002)
      [ADD]  -> 중복 없음 확인! RAG 본문 및 4-키 메타데이터 구성 완료.

    * 처리 대상 문서 #3 (ID: 2401.00003)
      [ADD]  -> 중복 없음 확인! RAG 본문 및 4-키 메타데이터 구성 완료.


## 4단계. 임베딩 모델 분기 및 캐시 어댑터 연결

선별된 도큐먼트들을 DB에 밀어 넣을 때, 3072차원의 벡터화 작업이 수반되어야 합니다.

### 💡 무비용 적재 어댑터 패턴의 필요성
LangChain의 `PGVector` 라이브러리는 `aadd_documents(batch)` 함수 호출 시 내부적으로 전달된 `Embeddings` 모델의 `embed_documents()` 메서드를 호출하도록 강제되어 있습니다. 
하지만 우리에게 이미 임베딩 캐시가 있다면 API 통신 및 요금을 지불할 필요가 없으므로, **LangChain의 규격을 준수하면서도 내부에서는 API를 쏘지 않고 순차적으로 캐시 데이터를 리턴해 주는 커스텀 어댑터**가 필요합니다.

In [9]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.embeddings import Embeddings

# backend/.env 파일 로드 (노트북 파일 위치 기준 backend/.env 찾기)
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정(.env)을 성공적으로 로드했습니다. (경로: {env_path})")

class OrderedPrecomputedEmbeddings(Embeddings):
    """전달되는 문서들의 임베딩을 캐싱된 임베딩 리스트에서 순서대로 꺼내어 반환하는 어댑터."""
    def __init__(self, embeddings: List[List[float]]):
        self.embeddings = embeddings
        self._index = 0

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        start = self._index
        end = start + len(texts)
        self._index = end
        print(f"        [OrderedPrecomputedEmbeddings] 배치 임베딩 요청 수령: {len(texts)}건 (인덱스 {start} ~ {end-1})")
        if start >= len(self.embeddings):
            return [[0.0] * 3072 for _ in texts]
        return self.embeddings[start:end]

    def embed_query(self, text: str) -> List[float]:
        return [0.0] * 3072

def resolve_embeddings_infrastructure(documents: List[Document], cached_embeddings: List[List[float] | None]):
    print(f"[STEP 4] 적재 대상 문서들의 임베딩 데이터 캐시 상태 진단...")
    has_all_embeddings = all(emb is not None and len(emb) == 3072 for emb in cached_embeddings)
    
    if has_all_embeddings and len(cached_embeddings) == len(documents):
        print(f"        -> [진단 결과: 합격] 모든 문서의 precomputed 캐시가 확인되어 OrderedPrecomputedEmbeddings 모드로 시작합니다.")
        valid_embeddings = [emb for emb in cached_embeddings if emb is not None]
        embeddings_impl = OrderedPrecomputedEmbeddings(valid_embeddings)
    else:
        print(f"        -> [진단 결과: 불합격] 캐시가 누락되었거나 불완전하여 OpenAI text-embedding-3-large 모델을 호출하여 임베딩을 신규 생성합니다.")
        api_key = os.getenv("OPENAI_API_KEY")
        embeddings_impl = OpenAIEmbeddings(
            model="text-embedding-3-large",
            api_key=api_key
        )
    return embeddings_impl

embeddings_impl = resolve_embeddings_infrastructure(documents, cached_embeddings)

✅ 백엔드 환경 설정(.env)을 성공적으로 로드했습니다. (경로: /Users/pileuszu/Repos/bist-mini-2/backend/.env)
[STEP 4] 적재 대상 문서들의 임베딩 데이터 캐시 상태 진단...
        -> [진단 결과: 불합격] 캐시가 누락되었거나 불완전하여 OpenAI text-embedding-3-large 모델을 호출하여 임베딩을 신규 생성합니다.


## 5단계. 통합 비동기 배치 적재 실행 시뮬레이션

마지막으로 가공 완료된 `Document` 리스트와 자동 판별된 임베딩 모듈을 결합하여, `BATCH_SIZE` 단위로 비동기 호출을 전송하는 적재 루프를 작동합니다.

여기서는 DB의 `langchain_pg_embedding` 테이블에 컬럼 데이터들이 각각 어떤 최종 형태로 조립되어 저장될지 자세하게 그 구조를 시뮬레이션 출력합니다.

In [10]:
async def run_unified_ingestion_dryrun(documents: List[Document], embeddings_adapter, batch_size: int = 1):
    print(f"[STEP 5] 표준 PGVector 비동기 배치 적재 시뮬레이션 시작...")
    
    batches = [documents[i:i + batch_size] for i in range(0, len(documents), batch_size)]
    total_loaded = 0
    
    for i, batch in enumerate(batches):
        print(f"\n========================================================================")
        print(f"  배치 업로드 실행 예정 #{i+1} (배치 내 도큐먼트 수: {len(batch)}건)")
        print(f"========================================================================")
        
        if isinstance(embeddings_adapter, OrderedPrecomputedEmbeddings):
            embeddings_list = embeddings_adapter.embed_documents([doc.page_content for doc in batch])
            engine_name = "OrderedPrecomputedEmbeddings 어댑터 (로컬 캐시)"
        else:
            embeddings_list = embeddings_adapter.embed_documents([doc.page_content for doc in batch])
            engine_name = "OpenAI API - text-embedding-3-large (API 실시간 생성)"
            
        for doc_idx, doc in enumerate(batch):
            vector_data = embeddings_list[doc_idx]
            
            print(f"\n  [DB langchain_pg_embedding 테이블 적재 예정 행 구조]")
            print(f"   1. [UUID] collection_id: 'langchain-collection-uuid-for-cs-embeddings'")
            print(f"   2. [TEXT] document (RAG 검색용 원문):\n      \"{doc.page_content}\"")
            print(f"   3. [JSONB] cmetadata (4-키 강제 표준 메타데이터):\n      {json.dumps(doc.metadata, indent=8, ensure_ascii=False)}")
            print(f"   4. [VECTOR] embedding (임베딩 엔진: {engine_name}):\n      {vector_data[:6]}... (총 {len(vector_data)}차원 벡터)")
            
        total_loaded += len(batch)
        
    print(f"\n========================================================================")
    print(f"[통합 완료] 총 {total_loaded}건의 신규 학술 논문이 BIST Mini-2 RAG pgvector DB에 적재 성공하였습니다!")
    print(f"========================================================================")

await run_unified_ingestion_dryrun(documents, embeddings_impl)

[STEP 5] 표준 PGVector 비동기 배치 적재 시뮬레이션 시작...

  배치 업로드 실행 예정 #1 (배치 내 도큐먼트 수: 1건)

  [DB langchain_pg_embedding 테이블 적재 예정 행 구조]
   1. [UUID] collection_id: 'langchain-collection-uuid-for-cs-embeddings'
   2. [TEXT] document (RAG 검색용 원문):
      "Title: Deep Learning in Biology

Abstract: This paper presents a new model for genetic analysis..."
   3. [JSONB] cmetadata (4-키 강제 표준 메타데이터):
      {
        "title": "Deep Learning in Biology",
        "arxiv_id": "2401.00001",
        "categories": "q-bio.GN",
        "update_date": ""
}
   4. [VECTOR] embedding (임베딩 엔진: OpenAI API - text-embedding-3-large (API 실시간 생성)):
      [0.0010957717895507812, 0.0016536712646484375, -0.01495361328125, -0.0066375732421875, 0.016815185546875, 0.0022640228271484375]... (총 3072차원 벡터)

  배치 업로드 실행 예정 #2 (배치 내 도큐먼트 수: 1건)

  [DB langchain_pg_embedding 테이블 적재 예정 행 구조]
   1. [UUID] collection_id: 'langchain-collection-uuid-for-cs-embeddings'
   2. [TEXT] document (RAG 검색용 원문):
      "Title: Sequence Alignment Al

## 6단계. 인덱스 최적화 및 운영 가이드라인

데이터 적재 완료 후 RAG 검색의 품질과 안정성을 확보하기 위해 다음 3가지 핵심 규칙을 항상 준수해 주십시오.

### 6.1 pgvector HNSW 인덱스 생성 및 파라미터 최적화
대량의 논문 데이터 임베딩(3072차원)에 대해 코사인 유사도 연산 속도와 정확도를 보장하기 위해 데이터베이스 단에서 반드시 **HNSW (Hierarchical Navigable Small World)** 인덱스를 설계해야 합니다.
```sql
CREATE INDEX IF NOT EXISTS langchain_pg_embedding_embedding_idx 
ON langchain_pg_embedding 
USING hnsw (embedding vector_cosine_ops)
WITH (m = 16, ef_construction = 64);
```
- `vector_cosine_ops`: 코사인 유사도 기반 거리 연산을 수행하는 옵션입니다.
- `m=16`, `ef_construction=64`: 인덱스 트리 구성 시 관계 복잡도와 빌드 정확도 간 균형을 맞추는 최적 권장 설정입니다.

### 6.2 3072차원 차원 매칭 보장
OpenAI의 최신 고해상도 임베딩 모델 `text-embedding-3-large`를 기준으로 사용하기 때문에, pgvector의 임베딩 컬럼 선언 차원은 반드시 **3072**차원으로 맞춰야 하며 다른 임베딩 모델(예: 1536차원의 text-embedding-ada-002)과 혼용되어 저장되지 않도록 철저히 통제되어야 합니다.

### 6.3 페이징(Paging) 조회 및 락업 방지
수만 건 이상의 대량 임베딩 데이터 백업/복원 또는 마이그레이션 도중 single query 호출로 인해 네트워크가 끊기거나 커넥션 락업(hang)이 걸리는 현상을 예방하기 위해, 데이터베이스 입출력 로직 설계 시 `LIMIT`와 `OFFSET`을 사용하는 페이징 처리를 항시 습관화하는 것이 중요합니다.